# Multimodal Ideal Point Model Comparison

This notebook compares six different approaches to estimating ideal points:
1. Model trained on votes only
2. Model trained on speeches only
3. Model trained on surveys only
4. Multimodal model on all three modalities with MoE gating
5. Multimodal model on all three modalities with PoE
6. Model trained on 1000 speeches, predicting out-of-sample for 9000 others

For each approach, we compute correlation with true ideal points and coverage rate of confidence intervals.

In [2]:
import sys
sys.path.append('../src/')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from deeplatent import generate_ideal_points, Corpus, IdealPointNN

# Set random seed for reproducibility
np.random.seed(42)

# Generate data for N observations
print("Generating data for N politicians...")
ideal_points, df, word_matrix, beta, label_coeffs = generate_ideal_points(
    num_politicians=10000,
    dim_ideal_points=1,
    num_bills=500,
    num_survey_questions=50,
    doc_length=100,
    vocab_size=500,
    progress_bar=True
)

print(f"Data generated: {df.shape[0]} politicians")
print(f"True ideal points range: [{ideal_points.min():.2f}, {ideal_points.max():.2f}]")

Generating data for N politicians...


Speeches:  12%| | 1201/10000 [00:13<01:39, 88.03it/s


KeyboardInterrupt: 

In [ ]:
# Prepare vectorizer for text
vectorizer = CountVectorizer()  # Limit features for computational efficiency
vectorizer.fit(df["doc_clean"])

# Function to create corpus with different modalities
def create_corpus(df, modalities_to_include):
    modalities = {}
    
    if "text" in modalities_to_include:
        modalities["text"] = {
            "column": "doc_clean",
            "views": {
                "bow": {
                    "type": "bow",
                    "vectorizer": vectorizer
                }
            }
        }
    
    if "vote" in modalities_to_include:
        modalities["vote"] = {
            "column": [f"vote_{i+1}" for i in range(1000)],
            "views": {
                "responses": {
                    "type": "vote"
                }
            }
        }
    
    if "survey" in modalities_to_include:
        modalities["survey"] = {
            "column": [f"Q_{i+1}" for i in range(50)],
            "views": {
                "responses": {
                    "type": "discrete_choice"
                }
            }
        }
    
    return Corpus(df, modalities=modalities)

print("Corpus creation function defined")

In [ ]:
# Function to train model and compute metrics
def train_and_evaluate(corpus, fusion_method="moe_average", model_name="Model", num_epochs=50):
    print(f"\nTraining {model_name}...")
    
    # Determine encoder/decoder arguments based on modalities
    encoder_args = {}
    decoder_args = {}
    
    for modality in corpus.modalities_config.keys():
        for view in corpus.modalities_config[modality]['views'].keys():
            key = f"{modality}_{view}"
            encoder_args[key] = {
                "hidden_dims": [128, 64],
                "activation": "relu",
                "bias": True,
                "dropout": 0.0
            }
            decoder_args[key] = {
                "hidden_dims": [],
                "activation": "relu",
                "bias": True,
                "dropout": 0.0
            }
    
    # Create and train model
    model = IdealPointNN(
        ae_type="vae",
        vi_type="full_rank",
        update_prior=False,
        n_ideal_points=1,
        batch_size=100,
        train_data=corpus,
        encoder_args=encoder_args,
        decoder_args=decoder_args,
        print_every_n_steps=100,
        num_steps=1000,
        w_prior=1,
        fusion=fusion_method
    )
    
    # Get predictions with uncertainty
    z = model.get_ideal_points(corpus, num_samples=100, return_samples=True)
    
    # Get initial means for affine alignment
    means_raw = z.mean(axis=1)  # [N, D] 
    est_raw = means_raw.flatten()
    true = ideal_points[:, 0].flatten()
    
    # Compute affine alignment parameters to map estimates to true values
    c = np.cov(true, est_raw, bias=True)[0,1] / np.var(est_raw, ddof=0)
    d = np.mean(true) - c * np.mean(est_raw)
    
    # Apply affine correction to all samples (align estimates to true values)
    z_corrected = c * z + d  # Apply to all samples [N, num_samples, D]
    
    # Compute corrected means and standard deviations
    means = z_corrected.mean(axis=1)  # [N, D]
    stds = z_corrected.std(axis=1)    # [N, D]
    
    # Flatten for analysis
    est = means.flatten()
    se = stds.flatten()
    
    # Compute correlation
    correlation = np.corrcoef(est, true)[0, 1]
    
    # Compute 95% CI coverage
    lower = est - 1.96 * se
    upper = est + 1.96 * se
    coverage = np.mean((true >= lower) & (true <= upper))
    
    print(f"{model_name} - Correlation: {correlation:.3f}, Coverage: {coverage:.3f}")
    
    return {
        'model': model,
        'estimated': est,
        'true_aligned': true,
        'correlation': correlation,
        'coverage': coverage,
        'std_errors': se
    }

print("Training and evaluation function defined")

In [ ]:
# 5. Train multimodal model with PoE
results['multimodal_poe'] = train_and_evaluate(
    full_corpus, 
    fusion_method="poe", 
    model_name="Multimodal (PoE)"
)

In [ ]:
# 6. Train on a fraction of speeches, predict out-of-sample for the others
print("\nTraining on a fraction of speeches, predicting out-of-sample for others...")

# Split data: first 1000 for training, remaining for testing
train_indices = np.arange(5000)
test_indices = np.arange(5000, 10000)

df_train = df.iloc[train_indices].reset_index(drop=True)
df_test = df.iloc[test_indices].reset_index(drop=True)

# Create train corpus
vectorizer_small = CountVectorizer()
vectorizer_small.fit(df_train["doc_clean"])

train_corpus = Corpus(df_train, modalities={
    "text": {
        "column": "doc_clean",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer_small
            }
        }
    }
})

# Train model on small dataset
model_oos = IdealPointNN(
    ae_type="vae",
    vi_type="full_rank",
    fixed_prior=True,
    n_ideal_points=1,
    batch_size=64,  
    train_data=train_corpus,
    encoder_args={
        "text_bow": {
            "hidden_dims": [128, 64],
            "activation": "relu",
            "bias": True,
            "dropout": 0.0
        }
    },
    decoder_args={
        "text_bow": {
            "hidden_dims": [],
            "activation": "relu",
            "bias": True,
            "dropout": 0.0
        }
    },
    print_every_n_epochs=1,
    print_every_n_batches=100,
    num_epochs=50,
    return_best_model=True,
    patience=5,
    w_prior=1,
    fusion="moe_average",
    kl_annealing_start=-1,
    kl_annealing_end=-1,
    free_bits=2.0
)

# Create test corpus with same vectorizer
test_corpus = Corpus(df_test, modalities={
    "text": {
        "column": "doc_clean",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer_small
            }
        }
    }
})

# Predict on test set
z_oos = model_oos.get_ideal_points(test_corpus, num_samples=100, return_samples=True)

# Get initial means for affine alignment
means_raw_oos = z_oos.mean(axis=1)  # [N, D] 
est_raw_oos = means_raw_oos.flatten()
true_oos = ideal_points[test_indices, 0].flatten()

# Compute affine alignment parameters to map estimates to true values
c_oos = np.cov(true_oos, est_raw_oos, bias=True)[0,1] / np.var(est_raw_oos, ddof=0)
d_oos = np.mean(true_oos) - c_oos * np.mean(est_raw_oos)

# Apply affine correction to all samples (align estimates to true values)
z_corrected_oos = c_oos * z_oos + d_oos  # Apply to all samples [N, num_samples, D]

# Compute corrected means and standard deviations
means_oos = z_corrected_oos.mean(axis=1)  # [N, D]
stds_oos = z_corrected_oos.std(axis=1)    # [N, D]

# Flatten for analysis
est_oos = means_oos.flatten()
se_oos = stds_oos.flatten()

# Compute correlation
correlation_oos = np.corrcoef(est_oos, true_oos)[0, 1]

# Compute 95% CI coverage
lower_oos = est_oos - 1.96 * se_oos
upper_oos = est_oos + 1.96 * se_oos
coverage_oos = np.mean((true_oos >= lower_oos) & (true_oos <= upper_oos))

results['out_of_sample'] = {
    'model': model_oos,
    'estimated': est_oos,
    'true_aligned': true_oos,
    'correlation': correlation_oos,
    'coverage': coverage_oos,
    'std_errors': se_oos
}

print(f"Out-of-sample - Correlation: {correlation_oos:.3f}, Coverage: {coverage_oos:.3f}")

In [ ]:
# Print summary of all results
print("\n" + "="*60)
print("SUMMARY OF RESULTS")
print("="*60)

model_names = {
    'votes_only': 'Votes Only',
    'speeches_only': 'Speeches Only', 
    'surveys_only': 'Surveys Only',
    'multimodal_moe': 'Multimodal (MoE Gating)',
    'multimodal_poe': 'Multimodal (PoE)',
    'out_of_sample': 'Out-of-Sample Speeches'
}

print(f"{'Model':<25} {'Correlation':<12} {'Coverage':<10}")
print("-"*47)

for key, name in model_names.items():
    corr = results[key]['correlation']
    cov = results[key]['coverage']
    print(f"{name:<25} {corr:<12.3f} {cov:<10.3f}")

In [ ]:
# Create the six-panel comparison plot
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

plot_titles = [
    'Votes Only',
    'Speeches Only', 
    'Surveys Only',
    'Multimodal (MoE Gating)',
    'Multimodal (PoE)',
    'Out-of-Sample Speeches'
]

plot_keys = ['votes_only', 'speeches_only', 'surveys_only', 
             'multimodal_moe', 'multimodal_poe', 'out_of_sample']

# Plot each comparison
for i, (key, title) in enumerate(zip(plot_keys, plot_titles)):
    ax = axes[i]
    
    # Get data
    estimated = results[key]['estimated']
    true_aligned = results[key]['true_aligned']
    correlation = results[key]['correlation']
    coverage = results[key]['coverage']
    
    # Sample points for cleaner visualization
    n_plot = min(1000, len(estimated))
    indices = np.random.choice(len(estimated), size=n_plot, replace=False)
    
    x_plot = true_aligned[indices]
    y_plot = estimated[indices]
    
    # Scatter plot
    ax.scatter(x_plot, y_plot, alpha=0.5, s=20, color='steelblue')
    
    # Linear fit
    coeffs = np.polyfit(x_plot, y_plot, 1)
    fit_line = np.poly1d(coeffs)
    x_range = np.linspace(x_plot.min(), x_plot.max(), 100)
    ax.plot(x_range, fit_line(x_range), 'r-', linewidth=2, alpha=0.8)
    
    # Identity line
    min_val = min(x_plot.min(), y_plot.min())
    max_val = max(x_plot.max(), y_plot.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=1)
    
    # Formatting
    ax.set_xlabel('True Ideal Points')
    ax.set_ylabel('Estimated Ideal Points')
    ax.set_title(title)
    
    # Add correlation and coverage text
    ax.text(0.05, 0.95, f'Correlation: {correlation:.3f}\nCoverage: {coverage:.3f}',
            transform=ax.transAxes, verticalalignment='top', 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            fontsize=10)
    
    # Make axes equal for better comparison
    ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.suptitle('Comparison of Ideal Point Estimation Methods', fontsize=16, y=1.02)
plt.show()

In [ ]:
# Additional analysis: Compare modality weights for multimodal models
print("\nModality Weights Analysis")
print("="*40)

# MoE Gating weights
if 'multimodal_moe' in results:
    print("\nMoE Gating Model:")
    moe_weights = results['multimodal_moe']['model'].get_modality_weights(full_corpus)
    moe_df = pd.DataFrame(moe_weights, columns=['text_bow', 'vote_responses', 'survey_responses'])
    print(f"Mean weights: {moe_df.mean()}")
    print(f"Std weights: {moe_df.std()}")

# PoE weights
if 'multimodal_poe' in results:
    print("\nPoE Model:")
    poe_weights = results['multimodal_poe']['model'].get_modality_weights(full_corpus)
    poe_df = pd.DataFrame(poe_weights, columns=['text_bow', 'vote_responses', 'survey_responses'])
    print(f"Mean weights: {poe_df.mean()}")
    print(f"Std weights: {poe_df.std()}")